# Energy Consumption Forecasting:
## Data Preprocessing, Feature Engineering, and Modeling Pipeline

This notebook processes the `household_power_consumption.txt` dataset and applies preprocessing steps derived from the EDA insights (`eda_infos.json`). Models trained: Ridge Regression, LightGBM, and XGBoost.

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

try:
    import xgboost as xgb
except ImportError:
    xgb = None

print("Libraries Loaded Successfully.")


### 1. Load Data and Metadata

In [ ]:
# Load EDA Insights
with open('eda_infos.json', 'r') as f:
    eda_infos = json.load(f)

# Load dataset
df = pd.read_csv('data/household_power_consumption.txt', sep=';', 
                 parse_dates={'datetime': ['Date', 'Time']}, 
                 infer_datetime_format=True, 
                 low_memory=False, 
                 na_values=['?'])

# Set index to datetime and sort (ensure temporal order)
df.set_index('datetime', inplace=True)
df.sort_index(inplace=True)

# Define components based on EDA
target = eda_infos['target_relationships']['primary_target']
# We include all predictive signals. Note that global_intensity is heavily correlated.
features = list(eda_infos['predictive_signal_ranking'].keys())
if target in features:
    features.remove(target)

print(f"Target: {target}")
print(f"Features: {features}")
print(f"Dataset shape before preprocessing: {df.shape}")


### 2. Preprocessing & Handling Missing Values
Based on EDA: Simple forward-fill combined with a rolling mean to respect temporal ordering, handling ~1.25% missing records.

In [ ]:
# Forward fill short gaps, then interpolate
df.ffill(limit=3, inplace=True)
# Fill remaining with short window rolling mean corresponding to previous 24 hours (given frequency is minutely, 24h = 1440 mins)
if df.isnull().sum().sum() > 0:
    df.fillna(df.rolling(window=1440, min_periods=1).mean(), inplace=True)
# Any remaining
df.fillna(method='bfill', inplace=True)
print("Missing values handled.")


### 3. Feature Engineering: Time, Lags, and Rolling Stats
Extracting cyclical patterns for 24H and 168H cycles, and Autoregressive lags.

In [ ]:
# 1. Cyclical Time Features
df['hour'] = df.index.hour
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month

# Sine/Cosine for Hour (24H cycle)
df['hour_sin'] = np.sin(2 * np.pi * df['hour']/23.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/23.0)

# Sine/Cosine for DayOfWeek (7 days cycle)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week']/6.0)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week']/6.0)

# 2. Lag Features for target (Assuming 1-step ahead is 1 minute. We will create lags for 1h, 24h, 1 week)
# To capture structural variance without leakage
# Assuming minutely data:
LAG_1H = 60
LAG_24H = 1440
LAG_1W = 10080

for col in features + [target]:
    df[f'{col}_lag1h'] = df[col].shift(LAG_1H)
    df[f'{col}_lag24h'] = df[col].shift(LAG_24H)

# Drop missing values induced by lags
df.dropna(inplace=True)

# Select final features
feature_cols = [c for c in df.columns if c != target and c not in ['hour', 'day_of_week', 'month']]
print(f"Engineered {len(feature_cols)} features.")


### 4. Transformations and Scaling
Applying `RobustScaler` mapped to Winter anomaly spikes. Applying `Yeo-Johnson` for heavy-tailed, highly skewed features like `Global_intensity` and `Sub_metering`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Identify skewed cols from EDA
skewed_cols = []
for k, v in eda_infos['distributions'].items():
    if v.get('requires_transformation', False) and k in feature_cols:
        skewed_cols.append(k)

print(f"Skewed columns for PowerTransformer: {skewed_cols}")

# Remaining columns go through RobustScaler
robust_cols = [c for c in feature_cols if c not in skewed_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('power', PowerTransformer(method='yeo-johnson'), skewed_cols),
        ('robust', RobustScaler(), robust_cols)
    ],
    remainder='passthrough'
)

# Wait, if we use target transformation, we might need TransformedTargetRegressor. 
# We'll transform target inside modeling or just manually.
if eda_infos['distributions'][target]['requires_transformation']:
    target_transformer = PowerTransformer(method='yeo-johnson')
    y_data = target_transformer.fit_transform(df[[target]])
else:
    target_transformer = None
    y_data = df[[target]].values
    
X_data = df[feature_cols]

# Temporal Train/Val/Test Split (80/10/10)
n = len(df)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

X_train, y_train = X_data.iloc[:train_end], y_data[:train_end]
X_val, y_val = X_data.iloc[train_end:val_end], y_data[train_end:val_end]
X_test, y_test = X_data.iloc[val_end:], y_data[val_end:]

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


### 5. Modeling
Training multiple regressors: `Ridge`, `LightGBM/HistGradientBoostingRegressor`, and `XGBoost`. Using strict temporal validation.

In [ ]:
# Create preprocessed data
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

models = {
    'Ridge': Ridge(alpha=1.0)
}

if lgb is not None:
    models['LightGBM'] = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.05, random_state=42)
else:
    models['HistGradientBoosting'] = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, random_state=42)

if xgb is not None:
    models['XGBoost'] = xgb.XGBRegressor(n_estimators=150, learning_rate=0.05, random_state=42, n_jobs=-1)

trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    # For LightGBM and XGBoost, we might want to use early stopping, but here we just fit to avoid complex API conditionals
    model.fit(X_train_processed, y_train.ravel())
    trained_models[name] = model

print("Training completed.")


### 6. Evaluation on Test Set

In [ ]:
results = {}
for name, model in trained_models.items():
    preds_transformed_scale = model.predict(X_test_processed)
    
    # inverse transform target if needed
    if target_transformer is not None:
        preds = target_transformer.inverse_transform(preds_transformed_scale.reshape(-1, 1)).ravel()
        y_test_inv = target_transformer.inverse_transform(y_test).ravel()
    else:
        preds = preds_transformed_scale
        y_test_inv = y_test.ravel()
        
    rmse = np.sqrt(mean_squared_error(y_test_inv, preds))
    mae = mean_absolute_error(y_test_inv, preds)
    r2 = r2_score(y_test_inv, preds)
    
    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }
    print(f"--- {name} ---")
    print(f"RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")


### 7. Feature Importances
Extracting feature importances from tree-based models.

In [ ]:
feature_importances = {}

# Reconstruct feature names after ColumnTransformer
transformed_skewed = skewed_cols
transformed_robust = robust_cols
passthrough_cols = [c for c in feature_cols if c not in skewed_cols and c not in robust_cols]
all_transformed_feature_names = transformed_skewed + transformed_robust + passthrough_cols

for name, model in trained_models.items():
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        # sort and get top 15
        indices = np.argsort(importances)[::-1][:15]
        top_features = {all_transformed_feature_names[i]: float(importances[i]) for i in indices}
        feature_importances[name] = top_features
    # Ridge model coefficients approximation
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_)
        indices = np.argsort(importances)[::-1][:15]
        top_features = {all_transformed_feature_names[i]: float(importances[i]) for i in indices}
        feature_importances[name] = top_features

from pprint import pprint
print("Top Predictive Signals extracted:")
pprint(feature_importances)


### 8. Exporting Metrics and Configurations
Saving the pipeline summary, final metrics, and applied transformations to JSON for MLOps compatibility.

In [ ]:
summary = {
    "model_performances": results,
    "feature_importances": feature_importances,
    "applied_preprocessing": {
        "missing_values": "Forward-fill + Rolling Mean 24h",
        "power_transformed_features": skewed_cols,
        "robust_scaled_features": robust_cols,
        "cyclical_features_added": ["hour_sin", "hour_cos", "dow_sin", "dow_cos"],
        "lag_features": ["1H", "24H"]
    },
    "target_transformer_used": "Yeo-Johnson" if target_transformer else "None",
    "evaluation_horizon": "1-minute step ahead (using historical lags natively)"
}

with open('pipeline_metrics_and_summary.json', 'w') as f:
    json.dump(summary, f, indent=4)

print("Pipeline metrics and summary saved to pipeline_metrics_and_summary.json")
